In [ ]:
from neo4j import GraphDatabase

# Function to connect to Neo4j
def connect_to_neo4j(uri, user, password):
    driver = GraphDatabase.driver(uri, auth=(user, password))
    return driver

# Connect to a Neo4j instance which enables Neo4j GDS, e. g. a local database
# adjust credentials according to your Neo4j instance
uri = "bolt://localhost:7689" 
user = "neo4j"
password = "password"
NEO4J_DB = "neo4j"

driver = connect_to_neo4j(uri, user, password)


In [2]:
# getting started with Neo4j Graph Data Science

from graphdatascience import GraphDataScience
gds = GraphDataScience(uri, auth=(user, password), database=NEO4J_DB)

# Check the installed GDS version on the server

print(gds.version())
assert gds.version() 

/home/ssc/projects/MI4People_Care_4_Rare_Five1/careforrare/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2.6.8


In [ ]:
## project training graph
## here, Biological_sample for training and test graph were not randomly sampled, but selected based on subjectid
## subjectid's may not be up-to-date anymore in newer dump-files for local databases
## -> please adjust the query accordingly or use the sampling approach in GDS_link_prediction_composite.py


G_train_exists = gds.run_cypher("""CALL gds.graph.exists("train_graph") YIELD exists""")

#pipe_cli_exists = gds.run_cypher("""CALL gds.pipeline.exists('pipe_cli') YIELD exists""")

if G_train_exists.iloc[0,0]==True:
    gds.graph.drop("train_graph")


#create graph projection
G_train, result = gds.graph.cypher.project("""MATCH (source)
    WHERE (source:Biological_sample AND source.subjectid STARTS WITH "10") OR
          (source:Biological_sample AND source.subjectid STARTS WITH "40") OR
          (source:Biological_sample AND source.subjectid STARTS WITH "41") OR
           source:Phenotype OR 
           source:Protein OR 
           source:Disease                                                                                                                  
    OPTIONAL MATCH (source)-[r:HAS_PHENOTYPE|HAS_DAMAGE|HAS_PARENT|HAS_PROTEIN|COMPILED_INTERACTS_WITH|HAS_DISEASE|IS_BIOMARKER_OF_DISEASE]->(target)
            WHERE target:Phenotype OR                                                                      
            target:Gene OR
            target:Protein OR
            target:Disease                               
    RETURN gds.graph.project(
    'train_graph',
    source,
    target,
    {
    sourceNodeLabels: labels(source),
    //sourceNodeProperties: source { .subjectid, .id},                                           
    targetNodeLabels: labels(target),
    //targetNodeProperties: target { .id},                                           
    relationshipType: type(r),
    relationshipProperties: r { .score }
    },
    { undirectedRelationshipTypes: ['HAS_DISEASE']}                                    
    )""")

assert G_train.node_count() == result["nodeCount"]

In [4]:
## project test graph

G_test_exists = gds.run_cypher("""CALL gds.graph.exists("test_graph") YIELD exists""")

#pipe_cli_exists = gds.run_cypher("""CALL gds.pipeline.exists('pipe_cli') YIELD exists""")

if G_test_exists.iloc[0,0]==True:
    gds.graph.drop("test_graph")

#create graph projection
G_test, result_test = gds.graph.cypher.project("""MATCH (source)
    WHERE (source:Biological_sample AND source.subjectid STARTS WITH "42") OR
          (source:Biological_sample AND source.subjectid STARTS WITH "43") OR
          (source:Biological_sample AND source.subjectid STARTS WITH "44") OR
           source:Phenotype OR
           source:Protein OR
           source:Disease                                                                                                                                  
    OPTIONAL MATCH (source)-[r:HAS_PHENOTYPE|HAS_DAMAGE|HAS_PROTEIN|COMPILED_INTERACTS_WITH|HAS_DISEASE|IS_BIOMARKER_OF_DISEASE]->(target)
            WHERE target:Gene OR target:Protein OR target:Phenotype OR
            (source:Biological_sample AND target:Disease AND target.id STARTS WITH "DOID:4")
    RETURN gds.graph.project(
    'test_graph',
    source,
    target,
    {
    sourceNodeLabels: labels(source),
    targetNodeLabels: labels(target),
    relationshipType: type(r),
    relationshipProperties: r { .score }
    },
    { undirectedRelationshipTypes: ['HAS_DISEASE']}                                    
    )""")
    
assert G_test.node_count() == result_test["nodeCount"]

In [5]:
#gds.graph.drop("train_graph")

gds.graph.list()

,degreeDistribution,graphName,database,databaseLocation,memoryUsage,sizeInBytes,nodeCount,relationshipCount,configuration,density,creationTime,modificationTime,schema,schemaWithOrientation
0,"{'min': 0, 'max': 1970, 'p90': 0, 'p999': 504,...",control_graph,neo4j,local,252 MiB,264855176,255581,1960445,"{'readConcurrency': 4, 'undirectedRelationship...",0.000030,2024-11-13T09:40:03.209570000+00:00,2024-11-13T09:40:50.395054000+00:00,"{'graphProperties': {}, 'nodes': {'Phenotype':...","{'graphProperties': {}, 'nodes': {'Phenotype':..."
1,"{'min': 0, 'max': 1970, 'p90': 0, 'p999': 504,...",test_graph,neo4j,local,182 MiB,191321656,255581,1959848,"{'readConcurrency': 4, 'undirectedRelationship...",0.000030,2024-11-13T10:57:21.009810000+00:00,2024-11-13T10:57:55.888718000+00:00,"{'graphProperties': {}, 'nodes': {'Phenotype':...","{'graphProperties': {}, 'nodes': {'Phenotype':..."
2,"{'min': 0, 'max': 1970, 'p90': 2, 'p999': 504,...",train_graph,neo4j,local,300 MiB,314827072,255795,1999483,"{'readConcurrency': 4, 'undirectedRelationship...",0.000031,2024-11-13T10:56:33.014197000+00:00,2024-11-13T10:57:20.955979000+00:00,"{'graphProperties': {}, 'nodes': {'Phenotype':...","{'graphProperties': {}, 'nodes': {'Phenotype':..."


##### approximate inductive link prediction using HashGNN for node embedding (Cypher)

In [6]:
if gds.run_cypher("""CALL gds.pipeline.exists('pipe_hashgnn') YIELD exists""").iloc[0,0]==True:
    gds.run_cypher("""CALL gds.pipeline.drop("pipe_hashgnn")""")

In [ ]:
# create pipeline
gds.beta.pipeline.linkPrediction.create('pipe_hashgnn')

# add node property
# for inductive link prediction with HashGNN, featureProperties and randomSeed are required
# use generateFeatures to create binary features (~ featureProperties)
# define contextNodeLabels and contextRelationshipTypes to facilitate model training
gds.run_cypher("""CALL gds.beta.pipeline.linkPrediction.addNodeProperty('pipe_hashgnn', 'hashgnn', {
               mutateProperty: 'embedding',
               iterations: 2,
               embeddingDensity: 512,
               heterogeneous: true,
               generateFeatures: {dimension: 12, densityLevel:2},   // HashGNN requires binary features
               contextNodeLabels: ['Protein', 'Gene', 'Phenotype'],
               contextRelationshipTypes: ['HAS_PROTEIN', 'HAS_DAMAGE', 'HAS_PHENOTYPE', 'HAS_PARENT', 'COMPILED_INTERACTS_WITH', 'IS_BIOMARKER_OF_DISEASE'],
               //outputDimension: 1,
               randomSeed: 123
               })""")


,name,nodePropertySteps,featureSteps,splitConfig,autoTuningConfig,parameterSpace
0,pipe_hashgnn,"[{'name': 'gds.hashgnn.mutate', 'config': {'em...",[],"{'testFraction': 0.1, 'validationFolds': 3, 't...",{'maxTrials': 10},"{'MultilayerPerceptron': [], 'RandomForest': [..."


In [ ]:
# add link features
gds.run_cypher(""" CALL gds.beta.pipeline.linkPrediction.addFeature('pipe_hashgnn', 'cosine', {
    nodeProperties: ['embedding']
})""")

#Configuring the relationship split -> what do you need the feature input for?
gds.run_cypher(""" CALL gds.beta.pipeline.linkPrediction.configureSplit('pipe_hashgnn', {
    testFraction: 0.2,
    trainFraction: 0.6,
    validationFolds: 3
})""")


# add model candidates
gds.run_cypher(""" CALL gds.beta.pipeline.linkPrediction.addLogisticRegression('pipe_hashgnn')""")
gds.run_cypher(""" CALL gds.beta.pipeline.linkPrediction.addRandomForest('pipe_hashgnn', {numberOfDecisionTrees: 100})""")
gds.run_cypher(""" CALL gds.alpha.pipeline.linkPrediction.addMLP('pipe_hashgnn', {hiddenLayerSizes: [64, 32], penalty: 0.01, patience: 2})""")


,name,nodePropertySteps,featureSteps,splitConfig,autoTuningConfig,parameterSpace
0,pipe_hashgnn,"[{'name': 'gds.hashgnn.mutate', 'config': {'em...","[{'name': 'HADAMARD', 'config': {'nodeProperti...","{'testFraction': 0.2, 'validationFolds': 3, 't...",{'maxTrials': 10},"{'MultilayerPerceptron': [{'minEpochs': 1, 'ma..."


In [9]:
if gds.run_cypher("""CALL gds.model.exists("pheno-hashgnn") YIELD exists""").iloc[0,0]==True:
    gds.run_cypher("""CALL gds.model.drop("pheno-hashgnn")""")

In [ ]:
# training
gds.run_cypher("""CALL gds.beta.pipeline.linkPrediction.train('train_graph', {
  pipeline: 'pipe_hashgnn',
  modelName: 'pheno-hashgnn',
  metrics: ['AUCPR', 'OUT_OF_BAG_ERROR'],
  sourceNodeLabel: 'Biological_sample',
  targetNodeLabel: 'Disease',
  targetRelationshipType: 'HAS_DISEASE',
  randomSeed: 42
}) YIELD modelInfo, modelSelectionStats
RETURN
  modelInfo.bestParameters AS winningModel,
  modelInfo.metrics.AUCPR.train.avg AS avgTrainScore,
  modelInfo.metrics.AUCPR.outerTrain AS outerTrainScore,
  modelInfo.metrics.AUCPR.test AS testScore,
  [cand IN modelSelectionStats.modelCandidates | cand.metrics.AUCPR.validation.avg] AS validationScores""")

## training only worked when defining contextNodeLabels and contextRelationshipTypes in addNodeProperty

,winningModel,avgTrainScore,outerTrainScore,testScore,validationScores
0,"{'minEpochs': 1, 'maxEpochs': 100, 'focusWeigh...",0.949776,0.931197,0.755245,"[0.8395124487387102, 0.8574583294400453, 0.863..."


In [11]:
predict_hashgnn = gds.run_cypher("""CALL gds.beta.pipeline.linkPrediction.predict.stream('test_graph', {
  modelName: 'pheno-hashgnn',
  topN: 500, 
  sampleRate: 1.0,         
  threshold: 0.1
})
 YIELD node1, node2, probability
 //WHERE NOT gds.util.asNode(node1).id STARTS WITH "HP:0000"
 //WHERE gds.util.asNode(node2).subjectid STARTS WITH "42066"
 //RETURN DISTINCT gds.util.asNode(node1).id AS HashGNN                                
 RETURN gds.util.asNode(node1).id AS disease_id, gds.util.asNode(node2).subjectid AS patient_id, probability
 //RETURN DISTINCT gds.util.asNode(node2).subjectid AS sample, COLLECT(DISTINCT gds.util.asNode(node1).id) AS disease, COUNT(DISTINCT gds.util.asNode(node1).id) AS count
 ORDER BY gds.util.asNode(node2).subjectid""")

predict_hashgnn

,disease_id,patient_id,probability
0,DOID:4372,42033,0.466102
1,DOID:6605,42033,0.338830
2,DOID:5285,42033,0.338830
3,DOID:14545,42033,0.338830
4,DOID:1519,42033,0.338830
...,...,...,...
495,DOID:746,44228,0.338830
496,DOID:5677,44228,0.338830
497,DOID:7147,44228,0.338830
498,DOID:1612,44228,0.338830


In [12]:
predict_hashgnn

,disease_id,patient_id,probability
0,DOID:4372,42033,0.466102
1,DOID:6605,42033,0.338830
2,DOID:5285,42033,0.338830
3,DOID:14545,42033,0.338830
4,DOID:1519,42033,0.338830
...,...,...,...
495,DOID:746,44228,0.338830
496,DOID:5677,44228,0.338830
497,DOID:7147,44228,0.338830
498,DOID:1612,44228,0.338830


In [ ]:
## compare predictions to actual data

ctrl = gds.run_cypher("""MATCH (bs:Biological_sample)
               WHERE bs.subjectid STARTS WITH "42" OR bs.subjectid STARTS WITH "43" OR bs.subjectid STARTS WITH "44"
               MATCH (bs)-[:HAS_DISEASE]->(d:Disease)
               RETURN d.id as disease_id, bs.subjectid as patient_id
               ORDER BY patient_id""")
ctrl

,disease_id,patient_id
0,DOID:10030,42075
1,DOID:6376,42075
2,DOID:10030,42135
3,DOID:4372,42199
4,DOID:112,42281
5,DOID:1680,42281
6,DOID:10030,42281
7,DOID:5295,42281
8,DOID:10230,42292
9,DOID:11963,42292


In [14]:
import pandas as pd

# Step 1: Merge DataFrames on both disease_id and patient_id
correct_predictions = pd.merge(predict_hashgnn, ctrl, on=['disease_id', 'patient_id'])

# Step 2: Identify false positives (predictions not in actual data)
false_positives = pd.merge(predict_hashgnn, correct_predictions, how='left', indicator=True)
false_positives = false_positives[false_positives['_merge'] == 'left_only'].drop(columns=['_merge'])

# Step 3: Identify false negatives (actual data not in predictions)
false_negatives = pd.merge(ctrl, correct_predictions, how='left', indicator=True)
false_negatives = false_negatives[false_negatives['_merge'] == 'left_only'].drop(columns=['_merge'])

# Display results
print("Correct Predictions:")
print(correct_predictions)

print("\nFalse Positives (Predicted but not actual):")
print(false_positives)

print("\nFalse Negatives (Actual but not predicted):")
print(false_negatives)

Correct Predictions:
Empty DataFrame
Columns: [disease_id, patient_id, probability]
Index: []

False Positives (Predicted but not actual):
     disease_id patient_id  probability
0     DOID:4372      42033     0.466102
1     DOID:6605      42033     0.338830
2     DOID:5285      42033     0.338830
3    DOID:14545      42033     0.338830
4     DOID:1519      42033     0.338830
..          ...        ...          ...
495    DOID:746      44228     0.338830
496   DOID:5677      44228     0.338830
497   DOID:7147      44228     0.338830
498   DOID:1612      44228     0.338830
499   DOID:5504      44228     0.338830

[500 rows x 3 columns]

False Negatives (Actual but not predicted):
      disease_id patient_id  probability
0     DOID:10030      42075          NaN
1      DOID:6376      42075          NaN
2     DOID:10030      42135          NaN
3      DOID:4372      42199          NaN
4       DOID:112      42281          NaN
5      DOID:1680      42281          NaN
6     DOID:10030      422

In [15]:
# Step 1: Group by patient_id and collect sets of disease_id
predicted_groups = predict_hashgnn.groupby('patient_id')['disease_id'].apply(set).reset_index()
actual_groups = ctrl.groupby('patient_id')['disease_id'].apply(set).reset_index()

# Step 2: Merge on patient_id to align predicted and actual disease sets per patient
merged_df = pd.merge(predicted_groups, actual_groups, on='patient_id', how='outer', suffixes=('_predicted', '_actual'))

# Step 3: Fill NaN values with empty sets
merged_df['disease_id_predicted'] = merged_df['disease_id_predicted'].apply(lambda x: x if isinstance(x, set) else set())
merged_df['disease_id_actual'] = merged_df['disease_id_actual'].apply(lambda x: x if isinstance(x, set) else set())

# Step 4: Check for any overlap in disease sets for each patient
merged_df['has_overlap'] = merged_df.apply(lambda row: bool(row['disease_id_predicted'] & row['disease_id_actual']), axis=1)

# Display results
print("Disease Prediction Comparison:")
print(merged_df[['patient_id', 'disease_id_predicted', 'disease_id_actual', 'has_overlap']])

Disease Prediction Comparison:
   patient_id                               disease_id_predicted  \
0       42033  {DOID:1519, DOID:5288, DOID:7147, DOID:1612, D...   
1       42066  {DOID:1519, DOID:5288, DOID:7147, DOID:1612, D...   
2       42075  {DOID:1519, DOID:5288, DOID:7147, DOID:1612, D...   
3       42135  {DOID:1519, DOID:5288, DOID:7147, DOID:1612, D...   
4       42199  {DOID:1519, DOID:5288, DOID:7147, DOID:1612, D...   
5       42231  {DOID:1519, DOID:5288, DOID:7147, DOID:1612, D...   
6       42275  {DOID:1519, DOID:5288, DOID:7147, DOID:1612, D...   
7       42281  {DOID:1519, DOID:5288, DOID:7147, DOID:1612, D...   
8       42292  {DOID:1519, DOID:5288, DOID:7147, DOID:1612, D...   
9       42302  {DOID:1519, DOID:5288, DOID:7147, DOID:1612, D...   
10      42321  {DOID:1519, DOID:5288, DOID:7147, DOID:1612, D...   
11      42346  {DOID:1519, DOID:5288, DOID:7147, DOID:1612, D...   
12      42367  {DOID:1519, DOID:5288, DOID:7147, DOID:1612, D...   
13      42412  {D

In [16]:
# Step 1: Extract unique disease IDs as sets
predicted_diseases = set(predict_hashgnn['disease_id'].unique())
actual_diseases = set(ctrl['disease_id'].unique())

# Step 2: Check for overlap
overlap = predicted_diseases & actual_diseases  # Intersection of both sets

# Results
if overlap:
    print("Overlap exists. The following disease IDs are present in both predicted and actual data:")
    print(overlap)
else:
    print("No overlap found between predicted and actual disease IDs.")

Overlap exists. The following disease IDs are present in both predicted and actual data:
{'DOID:4372'}
